# Two-dimensional dijet unfolding: beam-direction test

Train a flattened $p_{T}^{ave}$–$\eta_{CM}$ response on Pb-going embedding, apply it to the independent p-going embedding reco distribution, and compare the unfolded result with p-going generator truth. The response, training marginals, misses, fakes, and pair classification all come from Pb-going.

<!-- detailed-workflow-guide -->

### Detailed workflow and inverse problem

The pair $(p_T^{ave},\eta_{CM})$ is mapped to a global bin so the full response retains migrations in both coordinates. With $M_{ij}$ denoting matched events from truth bin $j$ to reco bin $i$, inclusive marginals satisfy $t_j=t_j^{matched}+t_j^{miss}$ and $m_i=m_i^{matched}+m_i^{fake}$.

Factorized unfolding applies $m_i^{signal}=P_im_i$ with $P_i=m_i^{matched}/m_i$, unfolds matched migrations with iterative Bayes, and returns inclusive truth through $\hat t_j=\hat t_j^{matched}/\epsilon_j$, $\epsilon_j=t_j^{matched}/t_j$. Forward folding reverses the physical mapping: the inclusive response reapplies efficiency and migration, then fakes are restored from the training fake-to-matched-reco ratio. Iterations regulate prior dependence versus variance; they do not guarantee unity for independent samples.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.

RooUnfold is loaded separately because only unfolding workflows require it.
Set `ROOUNFOLD_ROOT` when its checkout is not adjacent to this repository.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL
from hist_analysis.python.histogram_io import resolve_direction_file
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, save_canvas, set_2d_style, set_legend_style,
    set_pad_style, set_unfolding_1d_style,
)

# RooUnfold is optional and is initialized only for unfolding notebooks.
ROOUNFOLD_ROOT, ROOUNFOLD_LIBRARY = load_roounfold(
    ROOT,
    project_root=PROJECT_ROOT,
)

from hist_analysis.config.histograms import DIJET_PTAVE_BINS, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.unfolding import (
    UnfoldingInputKeys, as_pt_intervals, build_roounfold_response,
    calculate_response_diagnostics, flatten_pt_eta_projections,
    flatten_sparse_response, load_unfolding_inputs, project_eta_by_pt,
    project_response_eta_blocks, unfold_bayes, write_unfolding_output,
)
from hist_analysis.python.unfolding_plots import (
    draw_flattened_response, draw_projection_response,
    draw_unfolding_classification, draw_unfolding_closure,
    draw_unfolding_closure_by_pt,
)


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Set the ROOT style
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Load RooUnfold

In [ ]:
# Cell role: perform analysis step 3.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
try:
    import RooUnfold
except ImportError as exc:
    raise ImportError(
        f"Unable to import RooUnfold after loading {ROOUNFOLD_LIBRARY}. "
        f"Check that ROOUNFOLD_ROOT points to the RooUnfold checkout."
    ) from exc

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
TRAIN_DIRECTION = 'Pbgoing'  # response-training embedding direction
TEST_DIRECTION = 'pgoing'    # independent embedding direction to unfold
FILE_STEM = 'jetId'
# List of eta cuts for analysis
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
# Finite pT-average intervals used for the 2D unfolding. Values outside this range are excluded.
PTAVE_BIN_SET = 'test'  # test or standard
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}
PT_AVE_BINS = tuple(PTAVE_BIN_SETS[PTAVE_BIN_SET])
ETA_CUT_INDEX = 5
N_ITERATIONS = 20
MEASURED_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMJerDefExtraUnfold_{eta_cut_index}'
MEASURED_LABEL = 'Reco JER def.+#eta-dep.'
RESPONSE_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{eta_cut_index}'
MISS_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMMissJerDefExtra_{eta_cut_index}'
FAKE_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMFakeJerDefExtra_{eta_cut_index}'
CLASSIFICATION_HISTOGRAM_TEMPLATE = 'hUnfoldingPairClassificationJerDefExtra_{eta_cut_index}'
# Keep weighted response entries above RooUnfold's internal 1e-9 sanitization threshold.
RESPONSE_SCALE = 1.0e12
FLATTENED_RATIO_TO_GEN_Y_RANGE = (0.5, 2.5)
ETA_RATIO_TO_GEN_Y_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False

if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError(f'Unsupported GENERATOR={GENERATOR!r}')
for direction_name, direction in (('TRAIN_DIRECTION', TRAIN_DIRECTION), ('TEST_DIRECTION', TEST_DIRECTION)):
    if direction not in ('pgoing', 'Pbgoing'):
        raise ValueError(f'Unsupported {direction_name}={direction!r}')
if TRAIN_DIRECTION == TEST_DIRECTION:
    raise ValueError('TRAIN_DIRECTION and TEST_DIRECTION must be different for this test')
if RESPONSE_SCALE <= 0.0:
    raise ValueError(f'RESPONSE_SCALE must be positive, got {RESPONSE_SCALE}')
for range_name, y_range in (
    ('FLATTENED_RATIO_TO_GEN_Y_RANGE', FLATTENED_RATIO_TO_GEN_Y_RANGE),
    ('ETA_RATIO_TO_GEN_Y_RANGE', ETA_RATIO_TO_GEN_Y_RANGE),
):
    if len(y_range) != 2 or y_range[0] >= y_range[1]:
        raise ValueError(f'{range_name} must be an increasing (minimum, maximum) pair')

generator_label = GENERATOR.capitalize()
training_label = f'{generator_label} {TRAIN_DIRECTION} response'
test_label = f'{generator_label} {TEST_DIRECTION} test'
train_input_path = resolve_direction_file(BASE_DIR, GENERATOR, TRAIN_DIRECTION, FILE_STEM)
test_input_path = resolve_direction_file(BASE_DIR, GENERATOR, TEST_DIRECTION, FILE_STEM)
eta_cut_tag = f'{ETA_CUTS[ETA_CUT_INDEX]:g}'.replace('.', 'p')
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'unfold2D',
))
OUTPUT_TAG = (f'{GENERATOR}_{TRAIN_DIRECTION}_response_{TEST_DIRECTION}_test_'
              f'unfold2D_jerDefExtra_eta_{eta_cut_tag}_iter_{N_ITERATIONS}')
OUTPUT_ROOT_FILE = OUTPUT_DIR / f'{OUTPUT_TAG}.root'
eta_cuts = ETA_CUTS
pt_ave_bins = as_pt_intervals(PT_AVE_BINS)

generator_label

In [ ]:
# Cell role: resolve input files and load the named ROOT objects.
# Interpretation: Loaded objects that outlive their file must be cloned and detached from ROOT directories.
# The preceding Markdown gives the equations and physics assumptions for this step.
from hist_analysis.python.histogram_io import load_histogram
n_eta_cuts=len(eta_cuts); eta_idx=ETA_CUT_INDEX
training_keys=UnfoldingInputKeys(
    truth=f'hGenDijetPtEtaCM_{eta_idx}', measured=MEASURED_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    response=RESPONSE_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx), miss=MISS_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    fake=FAKE_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx), classification=CLASSIFICATION_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
)
training_inputs=load_unfolding_inputs(train_input_path,training_keys)
test_truth=load_histogram(str(test_input_path),f'hGenDijetPtEtaCM_{eta_idx}')
test_measured=load_histogram(str(test_input_path),MEASURED_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx))
genPtEtaCM=[None]*n_eta_cuts; genPtEtaCM[eta_idx]=test_truth
recoPtEtaCM=[None]*n_eta_cuts; recoPtEtaCM[eta_idx]=test_measured
trainingGenPtEtaCM=[None]*n_eta_cuts; trainingGenPtEtaCM[eta_idx]=training_inputs.truth
trainingRecoPtEtaCM=[None]*n_eta_cuts; trainingRecoPtEtaCM[eta_idx]=training_inputs.measured
genPtEtaCMMiss=[None]*n_eta_cuts; genPtEtaCMMiss[eta_idx]=training_inputs.miss
recoPtEtaCMFake=[None]*n_eta_cuts; recoPtEtaCMFake[eta_idx]=training_inputs.fake
genPtEtaCMVsRecoPtEtaCM=[None]*n_eta_cuts; genPtEtaCMVsRecoPtEtaCM[eta_idx]=training_inputs.response
unfoldingPairClassification=[None]*n_eta_cuts; unfoldingPairClassification[eta_idx]=training_inputs.classification
print(train_input_path,test_input_path);print(training_keys)

In [ ]:
# Cell role: draw the configured diagnostic figures and retain ROOT objects for display.
# Interpretation: Axis limits and logarithmic scales affect presentation only, not stored bin contents.
# The preceding Markdown gives the equations and physics assumptions for this step.
classification_canvas, classification = draw_unfolding_classification(
    unfoldingPairClassification[ETA_CUT_INDEX], annotations=(training_label, f'|#eta_{{CM}}^{{jet}}| < {ETA_CUTS[ETA_CUT_INDEX]:g}', MEASURED_LABEL),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_pair_classification.pdf', save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_unfolding_classification_jerDefExtra',
)
classification_canvas

In [ ]:
# Cell role: draw the configured diagnostic figures and retain ROOT objects for display.
# Interpretation: Axis limits and logarithmic scales affect presentation only, not stored bin contents.
# The preceding Markdown gives the equations and physics assumptions for this step.
# # For test purpose only, plot the first histogram

# if not genPtEtaCM:
#     raise RuntimeError("genPtEtaCM is empty; load the ROOT file first")

# canvas_name = "canvas_gen_pt_eta_cm_0"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_gen_dijet_pt_eta_cm_0 = ROOT.TCanvas(canvas_name, "cGenDijetPtEtaCM_0", 800, 800)
# canvas_gen_dijet_pt_eta_cm_0.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)
# ROOT.gPad.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
# set_2d_style(genPtEtaCM[0])
# genPtEtaCM[0].Draw("COLZ")
# canvas_gen_dijet_pt_eta_cm_0.Modified()
# canvas_gen_dijet_pt_eta_cm_0.Update()
# canvas_gen_dijet_pt_eta_cm_0

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
genEtaCM=[None]*n_eta_cuts; genEtaCM[eta_idx]=project_eta_by_pt(genPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hGenDijetEtaCM_{eta_idx}')
recoEtaCM=[None]*n_eta_cuts; recoEtaCM[eta_idx]=project_eta_by_pt(recoPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hRecoDijetEtaCM_{eta_idx}')
trainingGenEtaCM=[None]*n_eta_cuts; trainingGenEtaCM[eta_idx]=project_eta_by_pt(trainingGenPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hTrainingGenDijetEtaCM_{eta_idx}')
trainingRecoEtaCM=[None]*n_eta_cuts; trainingRecoEtaCM[eta_idx]=project_eta_by_pt(trainingRecoPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hTrainingRecoDijetEtaCM_{eta_idx}')
genEtaCMMiss=[None]*n_eta_cuts; genEtaCMMiss[eta_idx]=project_eta_by_pt(genPtEtaCMMiss[eta_idx],pt_ave_bins,name_prefix=f'hGenDijetEtaCMMiss_{eta_idx}')
recoEtaCMFake=[None]*n_eta_cuts; recoEtaCMFake[eta_idx]=project_eta_by_pt(recoPtEtaCMFake[eta_idx],pt_ave_bins,name_prefix=f'hRecoDijetEtaCMFake_{eta_idx}')
gen2recoResponse=[None]*n_eta_cuts; gen2recoResponse[eta_idx]=project_response_eta_blocks(genPtEtaCMVsRecoPtEtaCM[eta_idx],pt_ave_bins,name_prefix=f'hGen2RecoDijetEtaCM_{eta_idx}')

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
# # Plot all genEtaCM projections (all pt_ave bins) on one canvas for a given eta cut
# if not genEtaCM:
#     raise RuntimeError("genEtaCM is empty; run the projection cell first")

# eta_idx = 5  # choose eta-cut index here
# if eta_idx < 0 or eta_idx >= len(eta_cuts):
#     raise IndexError(f"eta_idx={eta_idx} is out of range for eta_cuts")

# canvas_name = f"canvas_genEtaCM_allPt_eta{eta_idx}"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_genEtaCM_allPt = ROOT.TCanvas(canvas_name, f"Gen etaCM projections (eta idx {eta_idx})", 800, 800)
# canvas_genEtaCM_allPt.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)

# gen_eta_overlays = []
# max_val = 0.0
# for pt_idx in range(len(pt_ave_bins) - 1):
#     hist = genEtaCM[eta_idx][pt_idx].Clone(f"hGenEtaCM_overlay_eta{eta_idx}_pt{pt_idx}")
#     hist.SetDirectory(0)
#     set_unfolding_1d_style(hist, 'gen')
#     integral = hist.Integral()
#     if integral > 0:
#         hist.Scale(1.0 / integral)
#     hist.SetTitle(";#eta_{CM};Normalized entries")
#     draw_opt = "E1" if pt_idx == 0 else "E1 SAME"
#     hist.Draw(draw_opt)
#     gen_eta_overlays.append(hist)
#     max_val = max(max_val, hist.GetMaximum())

# legend = ROOT.TLegend(0.6, 0.75, 0.88, 0.88)
# set_legend_style(legend)
# legend.SetTextFont(42)
# legend.SetTextSize(0.03)
# for pt_idx in range(len(pt_ave_bins) - 1):
#     label = f"{pt_ave_bins[pt_idx]} < p_{{T}}^{{ave}} < {pt_ave_bins[pt_idx + 1]} GeV"
#     legend.AddEntry(gen_eta_overlays[pt_idx], label, "p")
# legend.Draw()

# text = ROOT.TLatex()
# text.SetNDC(True)
# text.SetTextFont(42)
# text.SetTextSize(0.04)
# text.DrawLatex(0.16, 0.92, f"Gen #eta_{{CM}} projections, eta-cut index = {eta_idx}")

# canvas_genEtaCM_allPt.Modified()
# canvas_genEtaCM_allPt.Update()
# canvas_genEtaCM_allPt

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
eta_idx = ETA_CUT_INDEX
projection_response_canvases = []
for pt_bin_idx, (pt_low, pt_high) in enumerate(pt_ave_bins):
    canvas = draw_projection_response(
        {'gen': (f'{TRAIN_DIRECTION} Gen', trainingGenEtaCM[eta_idx][pt_bin_idx]),
         'reco': (f'{TRAIN_DIRECTION} {MEASURED_LABEL}', trainingRecoEtaCM[eta_idx][pt_bin_idx]),
         'miss': (f'{TRAIN_DIRECTION} Miss', genEtaCMMiss[eta_idx][pt_bin_idx]),
         'fake': (f'{TRAIN_DIRECTION} Fake', recoEtaCMFake[eta_idx][pt_bin_idx])},
        gen2recoResponse[eta_idx][pt_bin_idx], eta_range=(-eta_cuts[eta_idx]-0.1, eta_cuts[eta_idx]+0.1),
        response_titles=(f'{MEASURED_LABEL} #eta_{{CM}}', 'Gen #eta_{CM}'),
        annotations=(training_label, f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV', f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'),
        output=OUTPUT_DIR / f'{OUTPUT_TAG}_projection_response_pt_{pt_low:g}_{pt_high:g}.pdf', save_png=SAVE_PNG, grid=DRAW_GRID,
        canvas_name=f'canvas_projection_response_ptBin{pt_bin_idx}',
    )
    projection_response_canvases.append(canvas)
projection_response_canvases

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
eta_idx = ETA_CUT_INDEX
hGenTruthEtaCM, layout = flatten_pt_eta_projections(genEtaCM[eta_idx], name='hGenTruthEtaCM', pt_bins=pt_ave_bins)
hRecoMeasuredEtaCM, _ = flatten_pt_eta_projections(recoEtaCM[eta_idx], name='hRecoMeasuredEtaCM', layout=layout)
hTrainingTruthEtaCM, _ = flatten_pt_eta_projections(trainingGenEtaCM[eta_idx], name='hTrainingTruthEtaCM', layout=layout)
hTrainingMeasuredEtaCM, _ = flatten_pt_eta_projections(trainingRecoEtaCM[eta_idx], name='hTrainingMeasuredEtaCM', layout=layout)
hGenTruthEtaCMMiss, _ = flatten_pt_eta_projections(genEtaCMMiss[eta_idx], name='hGenTruthEtaCMMiss', layout=layout)
hRecoMeasuredEtaCMFake, _ = flatten_pt_eta_projections(recoEtaCMFake[eta_idx], name='hRecoMeasuredEtaCMFake', layout=layout)
hResponseEtaCM, _ = flatten_sparse_response(genPtEtaCMVsRecoPtEtaCM[eta_idx], pt_ave_bins, name='hResponseEtaCM', layout=layout)
nPtSelections, nEtaBins, nGlobalBins = layout.n_pt_bins, layout.n_eta_bins, layout.n_global_bins
print(f'nPtSelections={nPtSelections}, nEtaBins={nEtaBins}, nGlobalBins={nGlobalBins}')

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
canvas_flattened = draw_flattened_response(
    {'gen': (f'{TRAIN_DIRECTION} Gen', hTrainingTruthEtaCM), 'reco': (f'{TRAIN_DIRECTION} {MEASURED_LABEL}', hTrainingMeasuredEtaCM),
     'miss': (f'{TRAIN_DIRECTION} Miss', hGenTruthEtaCMMiss), 'fake': (f'{TRAIN_DIRECTION} Fake', hRecoMeasuredEtaCMFake)},
    hResponseEtaCM, annotations=(training_label, f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_flattened_response.pdf', save_png=SAVE_PNG, grid=DRAW_GRID, canvas_name='canvas_flattened',
)
canvas_flattened

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
diagnostics = calculate_response_diagnostics(
    hResponseEtaCM, hTrainingTruthEtaCM, hTrainingMeasuredEtaCM, explicit_miss=hGenTruthEtaCMMiss, explicit_fake=hRecoMeasuredEtaCMFake,
)
hMatchedTruthEtaCM, hMatchedRecoEtaCM = diagnostics.matched_truth, diagnostics.matched_measured
hEffectiveMissEtaCM, hEffectiveFakeEtaCM = diagnostics.effective_miss, diagnostics.effective_fake
hBoundaryMissEtaCM, hBoundaryFakeEtaCM = diagnostics.boundary_miss, diagnostics.boundary_fake
response_bundle = build_roounfold_response(
    RooUnfold, hTrainingTruthEtaCM, hTrainingMeasuredEtaCM, hResponseEtaCM, diagnostics=diagnostics, scale=RESPONSE_SCALE, require_fakes=True,
)
response = response_bundle.response
unfolding_result = unfold_bayes(RooUnfold, response_bundle, hRecoMeasuredEtaCM, iterations=N_ITERATIONS, handle_fakes=True, name='hUnfoldedEtaCM')
unfold, hUnfoldedEtaCM, covariance_matrix = unfolding_result.algorithm, unfolding_result.histogram, unfolding_result.covariance

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
canvas_unfolded, hMeasuredToTruth, hUnfoldedToTruth = draw_unfolding_closure(
    hGenTruthEtaCM, hRecoMeasuredEtaCM, hUnfoldedEtaCM, target_label='Gen', target_role='gen', measured_label=MEASURED_LABEL,
    x_title='global #eta_{CM} bin', ratio_range=FLATTENED_RATIO_TO_GEN_Y_RANGE, annotations=(test_label,),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_closure.pdf', save_png=SAVE_PNG, grid=DRAW_GRID, canvas_name='canvas_unfolded', ratio_name_prefix='hFlattenedClosure',
)
canvas_unfolded

## Unfolded eta distributions in all pTave intervals

Extract every pTave block from the flattened unfolded histogram. For each interval, compare unfolded, gen, and eta-dependent JER-default reco eta distributions and plot reco/gen and unfolded/gen ratios.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
(hUnfoldedEtaCMByPt, hRecoToGenEtaCMByPt, hUnfoldedToGenEtaCMByPt, unfolded_eta_canvases) = draw_unfolding_closure_by_pt(
    hUnfoldedEtaCM, genEtaCM[eta_idx], recoEtaCM[eta_idx], layout, output_dir=OUTPUT_DIR, output_tag=OUTPUT_TAG,
    target_label='Gen', target_role='gen', measured_label=MEASURED_LABEL, eta_range=(-eta_cuts[eta_idx]-0.1, eta_cuts[eta_idx]+0.1),
    ratio_range=ETA_RATIO_TO_GEN_Y_RANGE, annotation_prefix=(f'{training_label}; {test_label} spectra',), save_png=SAVE_PNG, grid=DRAW_GRID,
)
hGenEtaCMByPt = [h.Clone(f'hGenEtaCM_ptBin{i}') for i,h in enumerate(genEtaCM[eta_idx])]
hRecoEtaCMByPt = [h.Clone(f'hRecoEtaCM_ptBin{i}') for i,h in enumerate(recoEtaCM[eta_idx])]
unfolded_eta_canvases

## Save unfolding output

Write the flattened spectra, response diagnostics, unfolded result, ratios, covariance matrix, and configuration metadata to `hist_analysis/output/unfold2D/`.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
output_histograms=(hGenTruthEtaCM,hRecoMeasuredEtaCM,hTrainingTruthEtaCM,hTrainingMeasuredEtaCM,hGenTruthEtaCMMiss,hRecoMeasuredEtaCMFake,hResponseEtaCM,hMatchedTruthEtaCM,hMatchedRecoEtaCM,hEffectiveMissEtaCM,hEffectiveFakeEtaCM,hBoundaryMissEtaCM,hBoundaryFakeEtaCM,hUnfoldedEtaCM,hMeasuredToTruth,hUnfoldedToTruth,*hGenEtaCMByPt,*hRecoEtaCMByPt,*hUnfoldedEtaCMByPt,*hRecoToGenEtaCMByPt,*hUnfoldedToGenEtaCMByPt)
write_unfolding_output(OUTPUT_ROOT_FILE,histograms=output_histograms,covariance=covariance_matrix,response=response,metadata={'generator':GENERATOR,'train_direction':TRAIN_DIRECTION,'test_direction':TEST_DIRECTION,'pt_ave_bins':pt_ave_bins,'ptave_bin_set':PTAVE_BIN_SET,'eta_cut':eta_cuts[eta_idx],'iterations':N_ITERATIONS,'response_scale':RESPONSE_SCALE,'handle_fakes':True})
print(f'Wrote unfolding output to {OUTPUT_ROOT_FILE}')